<a href="https://colab.research.google.com/github/lmassaron/fine-tuning-workshop/blob/main/06_medical_synthetic_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop: Tunix-Med · Part 1: Building the Medical Mind

In this notebook, we automate the creation of a high-quality medical dataset focused on **Cardiology**. Domain adaptation requires data that is both **factually dense** and **properly formatted**.

### The Synthetic Data Strategy
Real-world clinical data is often private (HIPAA/GDPR). Synthetic data allows us to:
1.  **Extract Knowledge**: Convert unstructured text (Wikipedia) into structured Q&A.
2.  **Control Quality**: Use a "Teacher LLM" to generate and a "Judge LLM" to score the data.
3.  **Scale**: Generate thousands of high-quality examples in hours using **vLLM**.

### Pipeline Overview
1.  **Wikipedia Scraping**: Gathering raw cardiology text.
2.  **vLLM Acceleration**: Running a fast inference server to handle high-volume generation.
3.  **Create & Curate**: Using `synthetic-data-kit` to generate QA pairs and filter them based on a quality threshold (e.g., only keeping pairs with a score > 7).
4.  **Dataset Export**: Pushing the final dataset to the Hugging Face Hub.

In [1]:
import os
import re
import time
import subprocess
import wikipediaapi
import pandas as pd
from tqdm import tqdm
from datasets import Dataset, DatasetDict

# --- Workshop Settings ---
DEMO = False  # Set to True for a quick 5-minute run
MODEL_ENGINE = "Qwen/Qwen2.5-7B-Instruct"
NUM_PAIRS_PER_FILE = 30
QUALITY_THRESHOLD = 7.0


def init():
    """Scaffold the local data directory structure."""
    for d in ["medical_input", "generated", "curated", "final"]:
        os.makedirs(f"data/{d}", exist_ok=True)


init()

## 1 · Domain Scraping

We use the `wikipedia-api` to fetch content for a targeted list of 70+ cardiology topics. We clean the text to remove Wikipedia-specific artifacts (like citations) to ensure the LLM receives clean context.

### Tech Note: Knowledge Extraction
Domain adaptation works best when the model is exposed to **factually dense** text. By scraping specific Wikipedia sections (like Pathophysiology and Treatment), we provide the LLM with the raw material needed to generate high-quality clinical Q&A pairs.

In [ ]:
def clean_string(input_string):
    """Remove whitespace and bracketed citations."""
    cleaned_string = " ".join(input_string.split())
    cleaned_string = re.sub(r"\{.*?\}|\}", "", cleaned_string)
    return cleaned_string


def get_wikipedia_content(titles):
    """Crawl Wikipedia sections for relevant medical knowledge."""
    wiki_wiki = wikipediaapi.Wikipedia("MedicalBot/1.0", "en")
    extracted_texts = []
    for title in tqdm(titles):
        page = wiki_wiki.page(title)
        if page.exists():
            # Add the summary
            extracted_texts.append(f"{page.title} : {clean_string(page.summary)}")
            # Add non-trivial sections
            for section in page.sections:
                if len(section.text) > 200:
                    extracted_texts.append(
                        f"{page.title} ({section.title}) : {clean_string(section.text)}"
                    )
    return extracted_texts


# Extensive list of cardiology topics to build a comprehensive knowledge base
cardiology_topics = [
    "Hypertension", "Acute coronary syndrome", "Myocardial infarction", 
    "Stable angina", "Unstable angina", "Coronary artery disease", 
    "Coronary artery spasm", "TIMI risk score", "Percutaneous coronary intervention",
    "Coronary artery bypass surgery", "Heart failure",
    "Heart failure with preserved ejection fraction", "Dilated cardiomyopathy",
    "Hypertrophic cardiomyopathy", "Restrictive cardiomyopathy",
    "Takotsubo cardiomyopathy", "Cardiac resynchronization therapy",
    "Atrial fibrillation", "Atrial flutter", "Supraventricular tachycardia",
    "Ventricular tachycardia", "Ventricular fibrillation",
    "Wolff-Parkinson-White syndrome", "Long QT syndrome", "Brugada syndrome",
    "Sick sinus syndrome", "Atrioventricular block", "Cardiac pacemaker",
    "Implantable cardioverter-defibrillator", "Aortic stenosis",
    "Aortic regurgitation", "Mitral stenosis", "Mitral regurgitation",
    "Mitral valve prolapse", "Tricuspid regurgitation", "Infective endocarditis",
    "Transcatheter aortic valve replacement",
    "Pericarditis", "Cardiac tamponade", "Beck's triad (cardiac tamponade)",
    "Myocarditis", "Constrictive pericarditis", "Aortic aneurysm",
    "Aortic dissection", "Peripheral artery disease",
    "Deep vein thrombosis", "Pulmonary embolism", "Dyslipidemia",
    "Familial hypercholesterolemia", "Atherosclerosis", "Metabolic syndrome",
    "Congenital heart disease", "Atrial septal defect", "Ventricular septal defect", 
    "Tetralogy of Fallot", "Patent ductus arteriosus", "Pulmonary hypertension",
    "Cor pulmonale", "Electrocardiography", "Echocardiography", "Cardiac stress test",
    "Cardiac catheterization", "Holter monitor", "Troponin", 
    "B-type natriuretic peptide", "Beta blocker", "ACE inhibitor", 
    "Calcium channel blocker", "Statin", "Anticoagulant", "Antiarrhythmic agent",
    "Diuretic", "Digoxin", "Nitrate (pharmacology)",
]

if DEMO:
    cardiology_topics = cardiology_topics[:3]

extracted_texts = get_wikipedia_content(cardiology_topics)
for i, text in enumerate(extracted_texts):
    with open(f"data/medical_input/med_{i}.txt", "w") as f:
        f.write(text)

100%|██████████| 9/9 [00:04<00:00,  1.84it/s]


## 2 · vLLM Acceleration

Generating 10,000+ QA pairs would take days with standard library inference. We use **vLLM** (Variable Large Language Model) which uses **PagedAttention** to serve models at extreme speeds. We run it as a background process.

### Tech Note: The vLLM Command
The `vllm serve` command launches an OpenAI-compatible API. Key parameters used here:
- `--port 8000`: Standard port for API access.
- `--gpu-memory-utilization 0.25`: Limits VRAM usage to leave space for other tasks.
- `--max-model-len 8192`: Defines the context window size.
- `--enforce-eager`: Disables CUDA graphs to save memory during small-scale generation.

In [3]:
def start_vllm():
    """Start the vLLM OpenAI-compatible server in the background."""
    vllm_log = open("vllm_server.log", "w")
    return subprocess.Popen(
        [
            "vllm",
            "serve",
            MODEL_ENGINE,
            "--port",
            "8000",
            "--enforce-eager",
            "--gpu-memory-utilization",
            "0.25",
            "--max-model-len",
            "8192",
            "--disable-log-stats",
            "--uvicorn-log-level",
            "error",
        ],
        stdout=vllm_log,
        stderr=vllm_log,
    )


vllm_proc = start_vllm()
print("Waiting for vLLM server to be ready...")
while "Starting vLLM server" not in open("vllm_server.log").read():
    time.sleep(2)
print("vLLM server ready!")

Waiting for vLLM server to be ready...
vLLM server ready!


## 3 · Create & Curate Loop

We use `synthetic-data-kit` to automate the heavy lifting. The config below defines the medical system prompts and the grading rubric for the Judge LLM.

### Tech Note: The Synthetic Pipeline
The `synthetic-data-kit` follows a three-stage lifecycle for every input file:
1. **`create`**: Uses the "Teacher LLM" to generate raw Q&A pairs from text chunks.
2. **`curate`**: Uses the "Judge LLM" to rate each pair (1-10) and filters out low-quality data.
3. **`save-as --format ft`**: Converts the filtered JSON data into the standardized OpenAI Chat format (System/User/Assistant) required for fine-tuning.

In [4]:
# Define the YAML content using an f-string to inject the correct device
yaml_content = f"""
# Master configuration file for Synthetic Data Kit

# Global paths configuration
paths:
  # Input data locations
  input:
    txt: \"data/medical_input\"

  # Output locations
  output:
    parsed: \"data/output\"      # Where parsed text files are saved
    generated: \"data/generated\" # Where generated content is saved
    cleaned: \"data/cleaned\"     # Where cleaned content is saved
    final: \"data/final\"         # Where final formatted content is saved

# VLLM server configuration
vllm:
  api_base: \"http://localhost:8000/v1\" # Base URL for VLLM API
  port: 8000                           # Port for VLLM server
  model: \"{MODEL_ENGINE}\"              # Default model to use
  max_retries: 3                       # Number of retries for API calls
  retry_delay: 1.0                     # Initial delay between retries (seconds)

# Ingest configuration
ingest:
  default_format: \"txt\"  # Default output format for parsed files
  youtube_captions: \"auto\"  # Options: \"auto\", \"manual\" - caption preference

# LLM generation parameters
generation:
  temperature: 0.7        # Higher = more creative, lower = more deterministic
  top_p: 0.95             # Nucleus sampling parameter
  chunk_size: 1022        # Size of text chunks for processing
  overlap: 64             # Overlap between chunks to maintain context
  max_tokens: 1024        # Maximum tokens in LLM responses
  num_pairs: {NUM_PAIRS_PER_FILE}           # Default number of QA pairs to generate

# Content cleanup parameters
cleanup:
  threshold: {QUALITY_THRESHOLD}       # Default quality threshold (1-10)
  batch_size: 4        # Number of items per batch for rating
  temperature: 0.3     # Temperature for rating (lower = more consistent)

# Format conversion parameters
format:
  default: \"jsonl\"        # Default output format
  include_metadata: true  # Include metadata in output files
  pretty_json: true       # Use indentation in JSON output

# Prompts for different tasks
prompts:
  summary: |
    As an expert cardiologist, summarize this document in 7-10 sentences, focusing on the main topic, key concepts,
    key diagnostic criteria, relevant pathophysiology, and first-line treatment where applicable.
  qa_generation: |
    You are an expert cardiologist writing training data for a medical AI.
    You will be given a cardiology question and a short Wikipedia-style answer.
    Rewrite the answer to be clinically complete: include the key diagnostic criteria,
    relevant pathophysiology, and first-line treatment where applicable.
    Write 3-5 sentences. Do not add fabricated statistics. Stay factual.
    Text: {{{{text}}}}
  qa_rating: |
    As an expert cardiologist, rate these QA pairs as from 1-10 based on accuracy and clinical completeness.
    {{{{pairs}}}}
\"""

# Generate the YAML config for the synthetic-data-kit
with open("medical_data_config.yaml", "w") as f:
    f.write(yaml_content)

In [ ]:
input_files = sorted([
    os.path.join("data/medical_input", f)
    for f in os.listdir("data/medical_input")
    if f.endswith(".txt")
])
for filepath in tqdm(input_files):
    basename = os.path.basename(filepath).replace(".txt", "")
    gen_file = os.path.join("data/generated", f"{basename}_qa_pairs.json")
    curated_file = os.path.join("data/curated", f"{basename}_qa_pairs_cleaned.json")
    final_file = os.path.join("data/final", f"{basename}_qa_pairs_cleaned_ft.json")

    # Step 1: Create raw pairs
    if not os.path.exists(gen_file):
        subprocess.run(
            [
                "synthetic-data-kit",
                "-c",
                "medical_data_config.yaml",
                "create",
                filepath,
                "--type",
                "qa",
            ],
            check=True,
        )

    # Step 2: Curate (Judge) pairs
    if not os.path.exists(curated_file):
        if os.path.exists(gen_file):
            subprocess.run(
                [
                    "synthetic-data-kit",
                    "-c",
                    "medical_data_config.yaml",
                    "curate",
                    gen_file,
                ],
                check=True,
            )

    # Step 3: Save in Fine-Tuning (FT) format
    if not os.path.exists(final_file):
        if os.path.exists(curated_file):
            subprocess.run(
                [
                    "synthetic-data-kit",
                    "-c",
                    "medical_data_config.yaml",
                    "save-as",
                    curated_file,
                    "-f",
                    "ft",
                ],
                check=True,
            )

vllm_proc.terminate()
print("Generation and Curation complete!")

## 4 · Final Dataset Formatting\n
\n
We finalize the dataset by applying the standard **System Prompt** to every conversation. This ensures the model learns to associate the \"Knowledgeable medical assistant\" persona with accurate clinical output.

In [ ]:
SYSTEM_PROMPT = "You are a knowledgeable medical assistant specializing in cardiology. Answer clinical questions accurately, focusing on diagnostic criteria, treatment guidelines, and pathophysiology."


def set_system_prompt(row):
    if "messages" in row and len(row["messages"]) > 0:
        row["messages"][0]["content"] = SYSTEM_PROMPT
    return row


# Merge all cleaned files into a single dataframe
final_files = [n for n in os.listdir("data/final") if n.endswith("_ft.json")]
conversations = pd.concat(
    [pd.read_json(f"data/final/{n}") for n in final_files]
).reset_index(drop=True)
conversations = conversations.apply(set_system_prompt, axis=1)

complete_dataset = DatasetDict({"train": Dataset.from_pandas(conversations)})
complete_dataset.save_to_disk("data/medical-cardiology-qa")
print(f"Final dataset ready: {len(conversations)} cardiology clinical pairs.")

repository_id = "lmassaron/medical-cardiology-qa"
complete_dataset.push_to_hub(repository_id)
print(f"Final dataset pushed to Hugging Face @ {repository_id}")